# 🤖 CNN-LSTM BISINDO — Full Pipeline
**Skripsi:** Rancang Bangun Sistem Pengenalan Isyarat BISINDO Berbasis Sarung Tangan IoT Menggunakan CNN-LSTM

---
### 📋 Isi Notebook:
- **BAGIAN 0** — Setup (Mount Drive + Import)
- **BAGIAN 1** — Preprocessing & Validasi Data
- **BAGIAN 2** — Training Model CNN-LSTM
- **BAGIAN 3** — Evaluasi & Confusion Matrix
- **BAGIAN 4** — Simpan & Download File

> ⚠️ **Aktifkan GPU dulu: Runtime → Change runtime type → T4 GPU**

---
## ⚙️ BAGIAN 0 — Setup

In [ ]:
# ── Mount Google Drive ─────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os

# ============================================================
# SESUAIKAN PATH INI
# ============================================================
DRIVE_PATH   = '/content/drive/MyDrive/BISINDO_Skripsi'
CSV_FILE     = os.path.join(DRIVE_PATH, 'dataset_bisindo_mentah.csv')
DATASET_DIR  = os.path.join(DRIVE_PATH, 'dataset_processed')
MODEL_DIR    = os.path.join(DRIVE_PATH, 'model_output')
EVALUASI_DIR = os.path.join(DRIVE_PATH, 'evaluasi_output')

for d in [DATASET_DIR, MODEL_DIR, EVALUASI_DIR]:
    os.makedirs(d, exist_ok=True)

print('✅ Google Drive terhubung!')
print(f'📂 CSV      : {CSV_FILE}')
print(f'📂 Dataset  : {DATASET_DIR}')
print(f'📂 Model    : {MODEL_DIR}')
print(f'📂 Evaluasi : {EVALUASI_DIR}')

print('\nFile di folder BISINDO_Skripsi:')
try:
    for f in os.listdir(DRIVE_PATH):
        print(f'   {f}')
except:
    print('   ⚠️ Folder belum ada, buat dulu di Google Drive!')

In [ ]:
# ── Import Semua Library ───────────────────────────────────
import pandas as pd
import numpy as np
import joblib
import time
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score, precision_score, recall_score
)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv1D, BatchNormalization, MaxPooling1D,
    LSTM, Dropout, Dense, Input
)
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
)
from tensorflow.keras.utils import to_categorical

# Cek GPU
print(f'TensorFlow : {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'✅ GPU aktif: {gpus[0].name}')
else:
    print('⚠️ GPU tidak aktif! Ganti ke Runtime → T4 GPU')

# ── Konfigurasi Global ─────────────────────────────────────
TIMESTEPS         = 100
N_FEATURES        = 11
N_CLASSES         = 10
SAMPLES_PER_CLASS = 100

FEATURE_COLS = [
    'flex_1','flex_2','flex_3','flex_4','flex_5',
    'acc_x','acc_y','acc_z',
    'gyro_x','gyro_y','gyro_z'
]

LABEL_NAMES = [
    'Halo','Saya','Kamu','Terima Kasih','Sama-sama',
    'Maaf','Tolong','Sampai Jumpa','Apa Kabar','Baik'
]

print('\n✅ Semua library berhasil diimport!')
print(f'   Konfigurasi: {TIMESTEPS} timesteps × {N_FEATURES} fitur × {N_CLASSES} kelas')

---
## 📊 BAGIAN 1 — Preprocessing & Validasi Data
> Sesuai **Script 1** dari rancangan sistem

In [ ]:
print('=' * 55)
print('  STEP 1: LOAD & VALIDASI DATASET')
print('=' * 55)

df = pd.read_csv(CSV_FILE)
print(f'✅ File loaded: dataset_bisindo_mentah.csv')
print(f'   Total baris : {len(df):,}')
print(f'   Total kolom : {len(df.columns)}')
print(f'\n   Kolom: {list(df.columns)}')
print('\nContoh 3 baris pertama:')
display(df.head(3))

In [ ]:
# ── Validasi Distribusi ────────────────────────────────────
print('📊 Distribusi sampel per kelas:')
print('-' * 45)
counts = df.groupby('label')['sequence_id'].nunique()
for label, count in counts.items():
    status = '✅' if count == SAMPLES_PER_CLASS else '⚠️'
    bar = '█' * count + '░' * (SAMPLES_PER_CLASS - count)
    print(f'  {status} {label:15s}: {count:3d}/100 [{bar[:25]}]')
print(f'\n  Total sampel: {counts.sum()}/{N_CLASSES * SAMPLES_PER_CLASS}')

# ── Validasi Missing Values ────────────────────────────────
print('\n🔍 Cek missing values:')
missing = df[FEATURE_COLS].isnull().sum()
if missing.sum() == 0:
    print('  ✅ Tidak ada missing values!')
else:
    print(f'  ⚠️ Missing values ditemukan:\n{missing[missing > 0]}')

# ── Validasi Accel/Gyro ────────────────────────────────────
print('\n🔍 Cek nilai nol pada accel/gyro:')
for label in df['label'].unique():
    subset   = df[df['label'] == label]
    zero_rows = (subset[['acc_x','acc_y','acc_z']] == 0).all(axis=1).sum()
    status   = f'⚠️ {zero_rows} baris nol (cek parsing firmware!)' if zero_rows > 0 else '✅ Normal'
    print(f'  {label:15s}: {status}')

In [ ]:
print('\n' + '=' * 55)
print('  STEP 2: RESHAPE DATASET → (N, 100, 11)')
print('=' * 55)

X_list, y_list = [], []
skipped = 0

for seq_id, group in df.groupby('sequence_id'):
    if len(group) != TIMESTEPS:
        skipped += 1
        continue
    X_list.append(group[FEATURE_COLS].values)
    y_list.append(group['label'].iloc[0])

X = np.array(X_list, dtype=np.float32)
y = np.array(y_list)

print(f'  Shape X : {X.shape}  ← (sampel, timestep, fitur)')
print(f'  Shape y : {y.shape}')
if skipped > 0:
    print(f'  ⚠️ {skipped} sampel dibuang (frame tidak lengkap)')
else:
    print(f'  ✅ Semua sampel lengkap')

In [ ]:
print('\n' + '=' * 55)
print('  STEP 3: LABEL ENCODING')
print('=' * 55)

le = LabelEncoder()
y_encoded = le.fit_transform(y)
print('  Mapping kelas:')
for i, cls in enumerate(le.classes_):
    print(f'    {i} → {cls}')

joblib.dump(le, os.path.join(DATASET_DIR, 'label_encoder.pkl'))
print(f'\n  ✅ label_encoder.pkl disimpan')

In [ ]:
print('\n' + '=' * 55)
print('  STEP 4: STRATIFIED SPLIT 70/15/15')
print('=' * 55)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y_encoded, test_size=0.30, random_state=42, stratify=y_encoded)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print(f'  Training set   : {len(X_train)} sampel (70%)')
print(f'  Validation set : {len(X_val)} sampel (15%)')
print(f'  Test set       : {len(X_test)} sampel (15%)')

print('\n  Distribusi kelas per split:')
for name, y_s in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    unique, cts = np.unique(y_s, return_counts=True)
    print(f'  {name:5s}: {dict(zip(le.classes_[unique], cts))}')

In [ ]:
print('\n' + '=' * 55)
print('  STEP 5: Z-SCORE NORMALIZATION')
print('=' * 55)

_, t, f = X_train.shape

scaler = StandardScaler()
X_train_final = scaler.fit_transform(
    X_train.reshape(-1, f)).reshape(X_train.shape)
X_val_final   = scaler.transform(
    X_val.reshape(-1, f)).reshape(X_val.shape)
X_test_final  = scaler.transform(
    X_test.reshape(-1, f)).reshape(X_test.shape)

joblib.dump(scaler, os.path.join(DATASET_DIR, 'scaler.pkl'))
print(f'  ✅ scaler.pkl disimpan (fit dari training set saja)')
print(f'  Verifikasi → mean: {X_train_final.mean():.4f} (target ~0.0)')
print(f'             → std : {X_train_final.std():.4f} (target ~1.0)')

# One-hot
y_train_cat = to_categorical(y_train, N_CLASSES)
y_val_cat   = to_categorical(y_val,   N_CLASSES)
y_test_cat  = to_categorical(y_test,  N_CLASSES)

# Simpan numpy arrays
np.save(os.path.join(DATASET_DIR, 'X_train.npy'), X_train_final)
np.save(os.path.join(DATASET_DIR, 'X_val.npy'),   X_val_final)
np.save(os.path.join(DATASET_DIR, 'X_test.npy'),  X_test_final)
np.save(os.path.join(DATASET_DIR, 'y_train.npy'), y_train)
np.save(os.path.join(DATASET_DIR, 'y_val.npy'),   y_val)
np.save(os.path.join(DATASET_DIR, 'y_test.npy'),  y_test)

print(f'\n  ✅ Dataset processed tersimpan di: dataset_processed/')
print('\n' + '=' * 55)
print('  ✅ PREPROCESSING SELESAI!')
print('=' * 55)

---
## 🧠 BAGIAN 2 — Training Model CNN-LSTM
> Sesuai **Script 2** dari rancangan sistem

In [ ]:
print('=' * 55)
print('  STEP 1: MEMBANGUN ARSITEKTUR MODEL')
print('=' * 55)

LEARNING_RATE = 0.001
EPOCHS        = 100
BATCH_SIZE    = 32

model = Sequential([
    Input(shape=(TIMESTEPS, N_FEATURES)),

    # ── CNN Block ──────────────────────────────────────────
    Conv1D(filters=64,  kernel_size=3, padding='same', activation='relu'),
    BatchNormalization(),

    Conv1D(filters=128, kernel_size=3, padding='same', activation='relu'),
    BatchNormalization(),

    MaxPooling1D(pool_size=2, strides=2),

    # ── LSTM Block ─────────────────────────────────────────
    LSTM(128, return_sequences=True),
    LSTM(128, return_sequences=False),

    # ── Output Block ───────────────────────────────────────
    Dropout(0.3),
    Dense(N_CLASSES, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()
print(f'\n  Total parameter: {model.count_params():,}')

In [ ]:
print('\n' + '=' * 55)
print('  STEP 2: TRAINING')
print(f'  Epochs: {EPOCHS} | Batch: {BATCH_SIZE} | LR: {LEARNING_RATE}')
print('=' * 55)

callbacks = [
    ModelCheckpoint(
        filepath=os.path.join(MODEL_DIR, 'best_model.keras'),
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=7,
        min_lr=1e-6,
        verbose=1
    )
]

t_start = time.time()
history = model.fit(
    X_train_final, y_train_cat,
    validation_data=(X_val_final, y_val_cat),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)
t_end = time.time()

print(f'\n  ✅ Training selesai dalam {(t_end-t_start)/60:.1f} menit')
print(f'  Epoch berjalan   : {len(history.history["loss"])}')
print(f'  Best val_accuracy: {max(history.history["val_accuracy"]):.4f}')
print(f'  Best val_loss    : {min(history.history["val_loss"]):.4f}')

In [ ]:
# ── Training Curve ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('CNN-LSTM Training History — BISINDO', fontsize=13, fontweight='bold')

axes[0].plot(history.history['accuracy'],     label='Train', color='#2196F3', lw=2)
axes[0].plot(history.history['val_accuracy'], label='Val',   color='#FF9800', lw=2)
axes[0].axhline(y=0.85, color='red', linestyle='--', alpha=0.7, label='Target 85%')
axes[0].set_title('Accuracy'); axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy'); axes[0].legend()
axes[0].grid(alpha=0.3); axes[0].set_ylim([0, 1])

axes[1].plot(history.history['loss'],     label='Train', color='#2196F3', lw=2)
axes[1].plot(history.history['val_loss'], label='Val',   color='#FF9800', lw=2)
axes[1].set_title('Loss'); axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss'); axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
curve_path = os.path.join(MODEL_DIR, 'training_curve.png')
plt.savefig(curve_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'  ✅ Training curve disimpan → bahan Bab 4')

---
## 📈 BAGIAN 3 — Evaluasi & Confusion Matrix
> Sesuai **Script 3** dari rancangan sistem

In [ ]:
print('=' * 55)
print('  STEP 1: EVALUASI PADA TEST SET')
print('=' * 55)

best_model = tf.keras.models.load_model(
    os.path.join(MODEL_DIR, 'best_model.keras'))

test_loss, test_acc = best_model.evaluate(X_test_final, y_test_cat, verbose=0)
y_pred_prob = best_model.predict(X_test_final, verbose=0)
y_pred      = np.argmax(y_pred_prob, axis=1)

acc       = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='macro')
recall    = recall_score(y_test, y_pred, average='macro')
f1        = f1_score(y_test, y_pred, average='macro')

print(f'\n  ┌────────────────────────────────┐')
print(f'  │      HASIL EVALUASI AKHIR      │')
print(f'  ├────────────────────────────────┤')
print(f'  │  Accuracy  : {acc:.4f} ({acc*100:.2f}%)  │')
print(f'  │  Precision : {precision:.4f}             │')
print(f'  │  Recall    : {recall:.4f}             │')
print(f'  │  F1-Score  : {f1:.4f}             │')
print(f'  └────────────────────────────────┘')
print(f'\n  Target Akurasi ≥85%  : {"✅ TERCAPAI" if acc >= 0.85 else "⚠️ Belum tercapai"}')
print(f'  Target F1-Score ≥80% : {"✅ TERCAPAI" if f1  >= 0.80 else "⚠️ Belum tercapai"}')

In [ ]:
# ── Classification Report ──────────────────────────────────
print('\n📋 Classification Report per kelas:')
print('=' * 55)
report = classification_report(
    y_test, y_pred, target_names=LABEL_NAMES, digits=4)
print(report)

report_path = os.path.join(EVALUASI_DIR, 'classification_report.txt')
with open(report_path, 'w') as f:
    f.write('EVALUASI MODEL CNN-LSTM — BISINDO\n')
    f.write('=' * 55 + '\n\n')
    f.write(f'Accuracy  : {acc:.4f} ({acc*100:.2f}%)\n')
    f.write(f'Precision : {precision:.4f}\n')
    f.write(f'Recall    : {recall:.4f}\n')
    f.write(f'F1-Score  : {f1:.4f}\n\n')
    f.write('Per-Class Report:\n')
    f.write(report)
print(f'✅ Report disimpan → bahan Bab 5')

In [ ]:
# ── Confusion Matrix ───────────────────────────────────────
cm      = confusion_matrix(y_test, y_pred)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('Confusion Matrix — CNN-LSTM BISINDO', fontsize=13, fontweight='bold')

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, ax=axes[0])
axes[0].set_title('Raw Count')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
axes[0].tick_params(axis='x', rotation=45)

sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES,
            ax=axes[1], vmin=0, vmax=1)
axes[1].set_title('Normalized')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
cm_path = os.path.join(EVALUASI_DIR, 'confusion_matrix.png')
plt.savefig(cm_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Confusion matrix disimpan → bahan Bab 5')

# Analisis kesalahan
print('\n  Analisis kesalahan klasifikasi:')
for i, label in enumerate(LABEL_NAMES):
    total   = cm[i].sum()
    correct = cm[i][i]
    errors  = total - correct
    if errors > 0:
        wrong_idx = np.argmax([cm[i][j] if j != i else 0 for j in range(N_CLASSES)])
        print(f'  ⚠️ {label:15s}: {errors}/{total} salah → sering jadi "{LABEL_NAMES[wrong_idx]}"')
    else:
        print(f'  ✅ {label:15s}: sempurna (0 error)')

In [ ]:
# ── Pengujian Latensi Inferensi ────────────────────────────
print('\n' + '=' * 55)
print('  PENGUJIAN LATENSI INFERENSI')
print('=' * 55)

# Warm-up
_ = best_model.predict(X_test_final[:1], verbose=0)

latencies = []
for i in range(100):
    sample  = X_test_final[i:i+1]
    t0      = time.perf_counter()
    _       = best_model.predict(sample, verbose=0)
    latencies.append((time.perf_counter() - t0) * 1000)

latencies = np.array(latencies)
print(f'  Hasil 100 inferensi:')
print(f'  Rata-rata : {latencies.mean():.2f} ms')
print(f'  Minimum   : {latencies.min():.2f} ms')
print(f'  Maximum   : {latencies.max():.2f} ms')
print(f'  Std Dev   : {latencies.std():.2f} ms')
status = '✅ MEMENUHI' if latencies.mean() <= 100 else '⚠️ MELEBIHI'
print(f'\n  Target ≤100ms: {status} ({latencies.mean():.2f}ms)')

---
## 💾 BAGIAN 4 — Simpan & Download File

In [ ]:
# Simpan model final
model_final_path = os.path.join(MODEL_DIR, 'model_bisindo_final.keras')
best_model.save(model_final_path)

print('🎉 SEMUA FILE BERHASIL DISIMPAN KE GOOGLE DRIVE!')
print(f'\n📁 {DRIVE_PATH}/')
print('   ├── dataset_processed/')
print('   │   ├── X_train.npy, X_val.npy, X_test.npy')
print('   │   ├── y_train.npy, y_val.npy, y_test.npy')
print('   │   ├── scaler.pkl          ← untuk aplikasi desktop')
print('   │   └── label_encoder.pkl   ← untuk aplikasi desktop')
print('   ├── model_output/')
print('   │   ├── model_bisindo_final.keras  ← untuk aplikasi desktop')
print('   │   ├── best_model.keras')
print('   │   └── training_curve.png         ← bahan Bab 4')
print('   └── evaluasi_output/')
print('       ├── confusion_matrix.png        ← bahan Bab 5')
print('       └── classification_report.txt   ← bahan Bab 5')

print(f'\n📊 Ringkasan Hasil:')
print(f'   Accuracy  : {acc*100:.2f}%  {"✅" if acc >= 0.85 else "⚠️"}')
print(f'   F1-Score  : {f1*100:.2f}%  {"✅" if f1 >= 0.80 else "⚠️"}')
print(f'   Latensi   : {latencies.mean():.2f} ms  {"✅" if latencies.mean() <= 100 else "⚠️"}')

# Download 3 file penting ke PC lokal
from google.colab import files
print('\n⬇️  Download file untuk aplikasi desktop...')
files.download(model_final_path)
files.download(os.path.join(DATASET_DIR, 'scaler.pkl'))
files.download(os.path.join(DATASET_DIR, 'label_encoder.pkl'))
print('✅ Download selesai! Letakkan 3 file ini satu folder dengan aplikasi desktop.')